# MP6: Proyecto de IA y Big Data — Tema 2: Proyecto de Regresión

**Predicción de ventas diarias para un e-commerce (diciembre 2010 – diciembre 2011)**

---

## Planteamiento del problema

El objetivo es predecir el **valor total de ventas por día** entre el **9 de noviembre de 2011** y el **9 de diciembre de 2011** (test set), utilizando los datos históricos del 1 de diciembre de 2010 al 8 de noviembre de 2011 para entrenamiento y validación.

| Conjunto | Período |
|----------|--------|
| **Entrenamiento + Validación** | 01/12/2010 → 08/11/2011 |
| **Test** | 09/11/2011 → 09/12/2011 |

---

## Estructura del notebook

1. [Carga del dataset](#1-carga)
2. [Análisis Exploratorio (EDA)](#2-eda)
   - 2.1 Entendimiento de los datos
   - 2.2 Valores faltantes, duplicados y erróneos
   - 2.3 Búsqueda de outliers
   - 2.4 Gráficos auxiliares
   - 2.5 Análisis temporal
   - 2.6 Análisis de cancelaciones
   - 2.7 Ventas diarias brutas (variable objetivo)
   - 2.8 Top clientes y top productos
3. Limpieza de datos *(próxima sección)*
4. Transformación de datos *(próxima sección)*
5. División train / validation / test *(próxima sección)*
6. Modelos de regresión *(próxima sección)*

## 0. Imports y configuración global

In [ ]:
import os

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Configuración visual global
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

# Rutas
RUTA_CSV      = 'contenidoCSV/data.csv'
RUTA_GRAFICOS = 'graficos/'
os.makedirs(RUTA_GRAFICOS, exist_ok=True)

print('Librerías cargadas correctamente.')

---
<a id="1-carga"></a>
## 1. Carga del dataset

El dataset es el fichero `data.csv` descargado de Kaggle. Contiene las transacciones de un e-commerce del Reino Unido entre el 1 de diciembre de 2010 y el 9 de diciembre de 2011.  
Se carga con `read_csv` de Pandas especificando `encoding='latin-1'` para evitar errores con caracteres especiales.

In [ ]:
df = pd.read_csv(RUTA_CSV, encoding='latin-1')

print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
print('\nTipos de datos:')
print(df.dtypes)
print('\nPrimeras filas:')
df.head()

---
<a id="2-eda"></a>
## 2. Análisis Exploratorio de Datos (EDA)

Antes de limpiar o transformar nada, estudiamos a fondo el dataset para entender qué contiene cada columna, detectar anomalías y tomar decisiones informadas en la fase de limpieza.

### 2.1 Entendimiento de los datos

Exploramos dimensiones, tipos, valores únicos y estadísticas descriptivas. También identificamos transacciones con cantidades o precios negativos/cero, que en este dominio pueden corresponder a **devoluciones, regalos o ajustes contables**.

In [ ]:
print('--- 2.1.1 Dimensiones y tipos de datos ---')
print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
print('\nTipos de datos por columna:')
print(df.dtypes)

In [ ]:
print('--- 2.1.2 Primeras 10 filas ---')
display(df.head(10))

print('\n--- 2.1.3 Últimas 10 filas ---')
display(df.tail(10))

In [ ]:
print('--- 2.1.4 Valores únicos por columna ---')
for col in df.columns:
    print(f'  {col}: {df[col].nunique():,} valores únicos')

print('\n--- 2.1.5 Muestra de valores únicos (columnas categóricas) ---')
columnas_categoricas = ['InvoiceNo', 'StockCode', 'Description', 'Country']
for col in columnas_categoricas:
    muestra = df[col].dropna().unique()[:10]
    print(f'\n  {col} (primeros 10 únicos):')
    print(f'  {muestra}')

In [ ]:
print('--- 2.1.6 Estadísticas descriptivas (columnas numéricas) ---')
df.describe()

In [ ]:
print('--- 2.1.7 Transacciones con Quantity <= 0 ---')
trans_qty_negativa = df[df['Quantity'] <= 0]
print(f'  Total filas con Quantity <= 0: {len(trans_qty_negativa):,}')
print(f'  % sobre total: {len(trans_qty_negativa) / len(df) * 100:.2f}%')
display(trans_qty_negativa[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice']].head(10))

print('\n--- 2.1.8 Transacciones con UnitPrice <= 0 ---')
trans_price_negativa = df[df['UnitPrice'] <= 0]
print(f'  Total filas con UnitPrice <= 0: {len(trans_price_negativa):,}')
print(f'  % sobre total: {len(trans_price_negativa) / len(df) * 100:.2f}%')
display(trans_price_negativa[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice']].head(10))

### 2.2 Valores faltantes, duplicados y erróneos

Usamos `isnull`, `duplicated` y `value_counts` para cuantificar los problemas de calidad del dato:
- **CustomerID nulo**: transacciones anónimas (sin cliente registrado).
- **Description nula**: líneas de factura sin descripción de producto.
- **Filas duplicadas**: registros idénticos en todas sus columnas.
- **StockCodes no estándar**: códigos que no siguen el patrón `[0-9]{5}[A-Za-z]?` (p.ej. `POST`, `D`, `M`, `BANK CHARGES`).

In [ ]:
print('--- 2.2.1 Valores faltantes (NaN) por columna ---')
nulos     = df.isnull().sum()
nulos_pct = nulos / len(df) * 100
resumen_nulos = pd.DataFrame({'Nulos': nulos, '% sobre total': nulos_pct.round(2)})
display(resumen_nulos[resumen_nulos['Nulos'] > 0])

In [ ]:
print('--- 2.2.2 Filas sin CustomerID ---')
sin_cliente = df[df['CustomerID'].isnull()]
print(f'  Total: {len(sin_cliente):,} ({len(sin_cliente) / len(df) * 100:.2f}%)')
print('\n  Distribución por país (top 10):')
print(sin_cliente['Country'].value_counts().head(10))

In [ ]:
print('--- 2.2.3 Filas sin Description ---')
sin_descripcion = df[df['Description'].isnull()]
print(f'  Total: {len(sin_descripcion):,} ({len(sin_descripcion) / len(df) * 100:.2f}%)')
display(sin_descripcion[['InvoiceNo', 'StockCode', 'Quantity', 'UnitPrice', 'CustomerID']].head(10))

print('\n--- 2.2.4 Filas sin CustomerID y sin Description simultáneamente ---')
sin_ambos = df[df['CustomerID'].isnull() & df['Description'].isnull()]
print(f'  Total: {len(sin_ambos):,}')

In [ ]:
print('--- 2.2.5 Filas duplicadas (exactas) ---')
duplicados = df.duplicated()
print(f'  Total: {duplicados.sum():,} ({duplicados.sum() / len(df) * 100:.2f}%)')
display(df[duplicados].head(10))

In [ ]:
print('--- 2.2.6 Formato de InvoiceDate (actualmente string) ---')
print('  Muestra de valores de InvoiceDate:')
print(df['InvoiceDate'].value_counts().head(10))
print(f'\n  Min: {df["InvoiceDate"].min()}')
print(f'  Max: {df["InvoiceDate"].max()}')

In [ ]:
print('--- 2.2.7 StockCodes no estándar (posibles errores) ---')
# Patrón estándar: 5 dígitos + letra opcional
stock_no_estandar = df[~df['StockCode'].str.match(r'^[0-9]{5}[A-Za-z]?$', na=False)]
print(f'  Total filas con StockCode no estándar: {len(stock_no_estandar):,}')
print('  Tipos de StockCodes no estándar (top 15):')
print(stock_no_estandar['StockCode'].value_counts().head(15))

In [ ]:
print('--- 2.2.8 Distribución de Country (value_counts) ---')
print(df['Country'].value_counts())

### 2.3 Búsqueda de outliers

Aplicamos el **método IQR** (rango intercuartílico) tanto a `Quantity` como a `UnitPrice` para identificar valores extremos que podrían distorsionar el modelo. Como regla general, esperamos que el **90–95 % de los datos queden dentro del rango normal**.

$$\text{Límite inferior} = Q1 - 1{,}5 \cdot IQR \qquad \text{Límite superior} = Q3 + 1{,}5 \cdot IQR$$

In [ ]:
print('--- 2.3.1 Outliers en Quantity (método IQR) ---')
Q1_qty  = df['Quantity'].quantile(0.25)
Q3_qty  = df['Quantity'].quantile(0.75)
IQR_qty = Q3_qty - Q1_qty
lim_inf_qty = Q1_qty - 1.5 * IQR_qty
lim_sup_qty = Q3_qty + 1.5 * IQR_qty
outliers_qty = df[(df['Quantity'] < lim_inf_qty) | (df['Quantity'] > lim_sup_qty)]

print(f'  Q1={Q1_qty} | Q3={Q3_qty} | IQR={IQR_qty}')
print(f'  Límite inferior: {lim_inf_qty:.2f} | Límite superior: {lim_sup_qty:.2f}')
print(f'  Total outliers: {len(outliers_qty):,} ({len(outliers_qty) / len(df) * 100:.2f}%)')

print('\n  Top 5 mayores:')
display(df.nlargest(5, 'Quantity')[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice']])
print('\n  Top 5 menores:')
display(df.nsmallest(5, 'Quantity')[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice']])

In [ ]:
print('--- 2.3.2 Outliers en UnitPrice (método IQR) ---')
Q1_price  = df['UnitPrice'].quantile(0.25)
Q3_price  = df['UnitPrice'].quantile(0.75)
IQR_price = Q3_price - Q1_price
lim_inf_price = Q1_price - 1.5 * IQR_price
lim_sup_price = Q3_price + 1.5 * IQR_price
outliers_price = df[(df['UnitPrice'] < lim_inf_price) | (df['UnitPrice'] > lim_sup_price)]

print(f'  Q1={Q1_price} | Q3={Q3_price} | IQR={IQR_price}')
print(f'  Límite inferior: {lim_inf_price:.2f} | Límite superior: {lim_sup_price:.2f}')
print(f'  Total outliers: {len(outliers_price):,} ({len(outliers_price) / len(df) * 100:.2f}%)')

print('\n  Top 5 precios más altos:')
display(df.nlargest(5, 'UnitPrice')[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice']])

In [ ]:
print('--- 2.3.3 StockCodes menos frecuentes (outliers categóricos) ---')
stock_freq  = df['StockCode'].value_counts()
stock_raros = stock_freq[stock_freq <= 3]
print(f'  StockCodes con ≤3 apariciones: {len(stock_raros):,}')
print(stock_raros.head(10))

print('\n--- 2.3.4 Descriptions menos frecuentes (outliers categóricos) ---')
desc_freq  = df['Description'].value_counts()
desc_raras = desc_freq[desc_freq <= 3]
print(f'  Descriptions con ≤3 apariciones: {len(desc_raras):,}')
print(desc_raras.head(10))

In [ ]:
percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1.0]
print('--- 2.3.5 Distribución de Quantity por percentiles ---')
print(df['Quantity'].quantile(percentiles))

print('\n--- 2.3.6 Distribución de UnitPrice por percentiles ---')
print(df['UnitPrice'].quantile(percentiles))

### 2.4 Gráficos auxiliares

Visualizamos los hallazgos del EDA para facilitar su interpretación y documentación. Los gráficos se guardan además en `graficos/` para la memoria del proyecto.

In [ ]:
# 2.4.1 — Valores faltantes por columna
nulos_plot = df.isnull().sum()
nulos_plot = nulos_plot[nulos_plot > 0]

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=nulos_plot.index, y=nulos_plot.values, ax=ax)
ax.set_title('Valores faltantes por columna', fontsize=14)
ax.set_xlabel('Columna')
ax.set_ylabel('Nº de valores nulos')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 500,
            f'{int(bar.get_height()):,}',
            ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.4.1_valores_faltantes.png', dpi=150)
plt.show()

In [ ]:
# 2.4.2 — Distribución de Quantity (rango normal: 0–100)
qty_filtrado = df[(df['Quantity'] > 0) & (df['Quantity'] <= 100)]

fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(qty_filtrado['Quantity'], bins=50, ax=ax, kde=True)
ax.set_title('Distribución de Quantity (0–100 uds.)', fontsize=14)
ax.set_xlabel('Quantity')
ax.set_ylabel('Frecuencia')
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.4.2_distribucion_quantity.png', dpi=150)
plt.show()

In [ ]:
# 2.4.3 — Distribución de UnitPrice (rango normal: 0–20 £)
price_filtrado = df[(df['UnitPrice'] > 0) & (df['UnitPrice'] <= 20)]

fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(price_filtrado['UnitPrice'], bins=50, ax=ax, kde=True, color='coral')
ax.set_title('Distribución de UnitPrice (0–20 £)', fontsize=14)
ax.set_xlabel('UnitPrice (£)')
ax.set_ylabel('Frecuencia')
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.4.3_distribucion_unitprice.png', dpi=150)
plt.show()

In [ ]:
# 2.4.4 — Top 10 países por número de transacciones
top_paises = df['Country'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top_paises.values, y=top_paises.index, ax=ax, orient='h')
ax.set_title('Top 10 países por número de transacciones', fontsize=14)
ax.set_xlabel('Nº de transacciones')
ax.set_ylabel('País')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.4.4_top10_paises.png', dpi=150)
plt.show()

In [ ]:
# 2.4.5 — Proporción de transacciones por tipo de anomalía
total        = len(df)
n_qty_neg    = (df['Quantity'] <= 0).sum()
n_price_neg  = (df['UnitPrice'] <= 0).sum()
n_sin_client = df['CustomerID'].isnull().sum()
n_duplicados = df.duplicated().sum()
n_normales   = total - n_qty_neg - n_price_neg - n_sin_client - n_duplicados

labels  = ['Normales', 'Quantity ≤ 0', 'UnitPrice ≤ 0', 'Sin CustomerID', 'Duplicados']
valores = [n_normales, n_qty_neg, n_price_neg, n_sin_client, n_duplicados]
colores = ['#4CAF50', '#F44336', '#FF9800', '#2196F3', '#9C27B0']

fig, ax = plt.subplots(figsize=(8, 6))
ax.pie(valores, labels=labels, colors=colores, autopct='%1.1f%%', startangle=140)
ax.set_title('Proporción de transacciones por tipo de anomalía', fontsize=14)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.4.5_proporcion_anomalias.png', dpi=150)
plt.show()

In [ ]:
# 2.4.6 — Boxplots de Quantity y UnitPrice
qty_box   = df[(df['Quantity'] > 0)   & (df['Quantity'] <= 100)]['Quantity']
price_box = df[(df['UnitPrice'] > 0)  & (df['UnitPrice'] <= 20)]['UnitPrice']

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
sns.boxplot(y=qty_box,   ax=axes[0], color='steelblue')
axes[0].set_title('Boxplot Quantity (0–100)', fontsize=13)
axes[0].set_ylabel('Quantity')

sns.boxplot(y=price_box, ax=axes[1], color='coral')
axes[1].set_title('Boxplot UnitPrice (0–20 £)', fontsize=13)
axes[1].set_ylabel('UnitPrice (£)')

plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.4.6_boxplots.png', dpi=150)
plt.show()

### 2.5 Análisis temporal

Convertimos `InvoiceDate` a tipo `datetime` y analizamos la distribución de transacciones a lo largo del tiempo: por día, por mes y por día de la semana. También identificamos los **días sin datos** en el rango analizado.

In [ ]:
print('--- 2.5.1 Conversión de InvoiceDate a datetime ---')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='mixed')
print(f'  Tipo resultante : {df["InvoiceDate"].dtype}')
print(f'  Fecha mínima    : {df["InvoiceDate"].min()}')
print(f'  Fecha máxima    : {df["InvoiceDate"].max()}')
print(f'  Rango total     : {(df["InvoiceDate"].max() - df["InvoiceDate"].min()).days} días')

# Columna de fecha normalizada (sin hora)
df['Fecha']     = df['InvoiceDate'].dt.normalize()
df['Mes']       = df['InvoiceDate'].dt.to_period('M')
df['DiaSemana'] = df['InvoiceDate'].dt.day_name()

In [ ]:
print('--- 2.5.2 Transacciones por día ---')
trans_por_dia = df.groupby('Fecha').size().rename('NumTransacciones')
print(f'  Días con datos     : {len(trans_por_dia)}')
print(f'  Media trans/día    : {trans_por_dia.mean():.1f}')
print(f'  Máximo trans/día   : {trans_por_dia.max():,} ({trans_por_dia.idxmax().date()})')
print(f'  Mínimo trans/día   : {trans_por_dia.min():,} ({trans_por_dia.idxmin().date()})')

print('\n--- 2.5.3 Días sin datos en el rango ---')
rango_completo = pd.date_range(start=df['Fecha'].min(), end=df['Fecha'].max(), freq='D')
dias_sin_datos = rango_completo.difference(trans_por_dia.index)
print(f'  Días totales en el rango : {len(rango_completo)}')
print(f'  Días con datos           : {len(trans_por_dia)}')
print(f'  Días sin datos           : {len(dias_sin_datos)}')
print(dias_sin_datos.strftime('%Y-%m-%d').tolist())

In [ ]:
# 2.5.4 — Evolución de transacciones diarias
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(trans_por_dia.index, trans_por_dia.values, linewidth=1, color='steelblue')
ax.set_title('Evolución de transacciones diarias (dic 2010 – dic 2011)', fontsize=14)
ax.set_xlabel('Fecha')
ax.set_ylabel('Nº de transacciones')
ax.axvline(pd.Timestamp('2011-11-25'), color='orange', linestyle='--', linewidth=1.2, label='Black Friday 2011')
ax.axvline(pd.Timestamp('2011-12-01'), color='red',    linestyle='--', linewidth=1.2, label='Inicio diciembre (test set)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.5.4_transacciones_diarias.png', dpi=150)
plt.show()

In [ ]:
# 2.5.5 — Transacciones por mes
trans_por_mes = df.groupby('Mes').size().rename('NumTransacciones')

fig, ax = plt.subplots(figsize=(12, 5))
trans_por_mes.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Transacciones por mes', fontsize=14)
ax.set_xlabel('Mes')
ax.set_ylabel('Nº de transacciones')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.5.5_transacciones_por_mes.png', dpi=150)
plt.show()

In [ ]:
# 2.5.6 — Transacciones por día de la semana
dias_semana = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
trans_por_dia_semana = df.groupby('DiaSemana').size().reindex(dias_semana).rename('NumTransacciones')

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=trans_por_dia_semana.index, y=trans_por_dia_semana.values, ax=ax)
ax.set_title('Transacciones por día de la semana', fontsize=14)
ax.set_xlabel('Día')
ax.set_ylabel('Nº de transacciones')
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.5.6_transacciones_dia_semana.png', dpi=150)
plt.show()

### 2.6 Análisis de cancelaciones

Las facturas que empiezan por **`C`** en `InvoiceNo` son cancelaciones. Analizamos cuántas hay, cómo se cruzan con `Quantity < 0` y cuál es su evolución mensual.

Distinguimos tres categorías:
- **Cancelaciones normales**: prefijo `C` + `Quantity < 0`.
- **Cancelaciones con qty positiva**: prefijo `C` + `Quantity >= 0` (anomalía menor).
- **Huérfanos**: sin prefijo `C` pero con `Quantity < 0` (ajustes contables u otro tipo).

In [ ]:
print('--- 2.6.1 Facturas con prefijo C ---')
df['EsCancelacion'] = df['InvoiceNo'].str.startswith('C')
n_cancelaciones = df['EsCancelacion'].sum()
print(f'  Total filas con prefijo C       : {n_cancelaciones:,}')
print(f'  % sobre total                   : {n_cancelaciones / len(df) * 100:.2f}%')
print(f'  Facturas únicas canceladas      : {df[df["EsCancelacion"]]["InvoiceNo"].nunique():,}')

In [ ]:
print('--- 2.6.2 Cruce entre prefijo C y Quantity < 0 ---')
con_C_qty_neg = df[ df['EsCancelacion'] &  (df['Quantity'] < 0)]
con_C_qty_pos = df[ df['EsCancelacion'] & (df['Quantity'] >= 0)]
sin_C_qty_neg = df[~df['EsCancelacion'] &  (df['Quantity'] < 0)]
sin_C_qty_pos = df[~df['EsCancelacion'] & (df['Quantity'] >= 0)]

print(f'  Prefijo C + Qty<0  (cancelaciones normales)     : {len(con_C_qty_neg):>7,}')
print(f'  Prefijo C + Qty>=0 (cancelaciones qty positiva) : {len(con_C_qty_pos):>7,}')
print(f'  Sin C   + Qty<0    (huérfanos)                  : {len(sin_C_qty_neg):>7,}')
print(f'  Sin C   + Qty>=0   (transacciones normales)     : {len(sin_C_qty_pos):>7,}')

In [ ]:
print('--- 2.6.3 Detalle de negativos huérfanos (sin prefijo C) ---')
print(f'  Total: {len(sin_C_qty_neg):,}')
if len(sin_C_qty_neg) > 0:
    display(sin_C_qty_neg[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice', 'CustomerID']].head(10))
    print('\n  StockCodes más frecuentes en huérfanos:')
    print(sin_C_qty_neg['StockCode'].value_counts().head(10))

print('\n--- 2.6.4 Cancelaciones con Quantity >= 0 (anomalía) ---')
print(f'  Total: {len(con_C_qty_pos):,}')
if len(con_C_qty_pos) > 0:
    display(con_C_qty_pos[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice']].head(10))

In [ ]:
print('--- 2.6.5 Cancelaciones por mes ---')
cancelaciones_mes = df[df['EsCancelacion']].groupby('Mes').size().rename('Cancelaciones')
total_mes         = df.groupby('Mes').size().rename('Total')
ratio_cancelacion = (cancelaciones_mes / total_mes * 100).round(2).rename('% Cancelaciones')
display(pd.concat([total_mes, cancelaciones_mes, ratio_cancelacion], axis=1))

In [ ]:
# 2.6.6 — Proporción cancelaciones vs normales
datos_grafico = {
    'Normales':                  len(sin_C_qty_pos),
    'Cancelaciones (C + Qty<0)': len(con_C_qty_neg),
    'Huérfanos (sin C + Qty<0)': len(sin_C_qty_neg),
}
labels_c  = list(datos_grafico.keys())
valores_c = list(datos_grafico.values())
colores_c = ['#4CAF50', '#F44336', '#9C27B0']

fig, ax = plt.subplots(figsize=(9, 7))
wedges, texts, autotexts = ax.pie(
    valores_c,
    colors=colores_c,
    autopct=lambda p: f'{p:.1f}%\n({int(p * sum(valores_c) / 100):,})',
    startangle=140,
    pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for autotext in autotexts:
    autotext.set_fontsize(10)
ax.legend(wedges, labels_c, loc='lower center', bbox_to_anchor=(0.5, -0.08), ncol=1, fontsize=10)
ax.set_title('Tipos de transacciones: normales vs cancelaciones', fontsize=13, pad=20)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.6.6_cancelaciones_proporcion.png', dpi=150)
plt.show()

In [ ]:
# 2.6.7 — Tasa de cancelación por mes
fig, ax = plt.subplots(figsize=(12, 5))
ratio_cancelacion.plot(kind='bar', ax=ax, color='salmon', edgecolor='white')
ax.set_title('Tasa de cancelación mensual (%)', fontsize=14)
ax.set_xlabel('Mes')
ax.set_ylabel('% cancelaciones sobre total')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.6.7_tasa_cancelacion_mensual.png', dpi=150)
plt.show()

### 2.7 Ventas diarias brutas — Variable objetivo

Calculamos `TotalPrice = Quantity × UnitPrice` para obtener el valor económico de cada línea de transacción. Luego **agregamos por día** para tener una primera versión (sin limpiar) de la variable objetivo: **ventas totales diarias en £**.

> Nota: esta agregación es bruta, incluye cancelaciones y valores negativos. La versión limpia se generará en la siguiente sección.

In [ ]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print('--- 2.7.1 Estadísticas de TotalPrice por fila ---')
print(df['TotalPrice'].describe())

In [ ]:
print('--- 2.7.2 Agregación de ventas por día (bruto) ---')
ventas_diarias = df.groupby('Fecha')['TotalPrice'].sum().rename('VentasDiarias')
print(f'  Días con datos            : {len(ventas_diarias)}')
print(f'  Venta media diaria        : £{ventas_diarias.mean():>12,.2f}')
print(f'  Venta mediana diaria      : £{ventas_diarias.median():>11,.2f}')
print(f'  Venta máxima diaria       : £{ventas_diarias.max():>11,.2f} ({ventas_diarias.idxmax().date()})')
print(f'  Venta mínima diaria       : £{ventas_diarias.min():>11,.2f} ({ventas_diarias.idxmin().date()})')
print(f'  Días con ventas negativas : {(ventas_diarias < 0).sum()}')

In [ ]:
print('--- 2.7.3 Ventas totales por mes ---')
ventas_mes = df.groupby('Mes')['TotalPrice'].sum().rename('VentasMes')
print(ventas_mes.apply(lambda x: f'£{x:,.2f}'))

print('\n--- 2.7.4 Distribución de ventas diarias por percentiles ---')
percentiles_v = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
print(ventas_diarias.quantile(percentiles_v).apply(lambda x: f'£{x:,.2f}'))

In [ ]:
# 2.7.5 — Evolución de ventas diarias brutas
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ventas_diarias.index, ventas_diarias.values, linewidth=1, color='steelblue')
ax.axhline(ventas_diarias.mean(), color='orange', linestyle='--', linewidth=1.2,
           label=f'Media: £{ventas_diarias.mean():,.0f}')
ax.axvline(pd.Timestamp('2011-11-09'), color='green', linestyle='--', linewidth=1.2,
           label='Inicio test set (9 nov 2011)')
ax.set_title('Evolución de ventas diarias brutas — variable objetivo (sin limpiar)', fontsize=13)
ax.set_xlabel('Fecha')
ax.set_ylabel('Ventas (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.7.5_ventas_diarias_brutas.png', dpi=150)
plt.show()

In [ ]:
# 2.7.6 — Ventas totales por mes
fig, ax = plt.subplots(figsize=(12, 5))
ventas_mes.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Ventas totales brutas por mes (£)', fontsize=14)
ax.set_xlabel('Mes')
ax.set_ylabel('Ventas (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e6:.1f}M'))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.7.6_ventas_mensuales_brutas.png', dpi=150)
plt.show()

In [ ]:
# 2.7.7 — Distribución de ventas diarias (días > 0)
ventas_positivas = ventas_diarias[ventas_diarias > 0]

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(ventas_positivas, bins=40, ax=ax, kde=True, color='steelblue')
ax.axvline(ventas_positivas.mean(),   color='orange', linestyle='--', linewidth=1.5,
           label=f'Media: £{ventas_positivas.mean():,.0f}')
ax.axvline(ventas_positivas.median(), color='green',  linestyle='--', linewidth=1.5,
           label=f'Mediana: £{ventas_positivas.median():,.0f}')
ax.set_title('Distribución de ventas diarias brutas (días con ventas > 0)', fontsize=13)
ax.set_xlabel('Ventas (£)')
ax.set_ylabel('Frecuencia')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.savefig(f'{RUTA_GRAFICOS}2.7.7_distribucion_ventas_diarias.png', dpi=150)
plt.show()

### 2.8 Top clientes y top productos

Analizamos qué clientes y productos concentran la mayor parte de las ventas. Es relevante para detectar posibles outliers de negocio (un cliente mayorista con miles de unidades puede distorsionar los patrones de ventas minoristas).

In [ ]:
print('--- 2.8.1 Top 10 clientes por volumen de ventas ---')
ventas_cliente = (
    df[df['CustomerID'].notna()]
    .groupby('CustomerID')['TotalPrice']
    .sum()
    .sort_values(ascending=False)
)
top10_clientes = ventas_cliente.head(10)
total_ventas   = df['TotalPrice'].sum()

print(f'  Ventas totales brutas: £{total_ventas:,.2f}')
print(f'\n  {"CustomerID":<15} {"Ventas (£)":>15} {"% sobre total":>15}')
print(f'  {"-"*45}')
for cid, ventas in top10_clientes.items():
    print(f'  {int(cid):<15} £{ventas:>13,.2f} {ventas / total_ventas * 100:>14.2f}%')

pct_top10 = top10_clientes.sum() / total_ventas * 100
print(f'\n  Top 10 clientes concentran el {pct_top10:.1f}% de las ventas totales')

# 3. LIMPIEZA DE DATOS

In [ ]:
df_clean = df.copy()
filas_iniciales = len(df_clean)
print(f"\n\n{'='*60}")
print(f"  INICIO LIMPIEZA — Filas iniciales: {filas_iniciales:,}")
print(f"{'='*60}")

### 3.1 ELIMINAR FILAS CON Description NULA

Motivo: el 100% de estas filas cumplen simultáneamente:
 - Description = NaN  → no sabemos qué producto es
 - UnitPrice = 0      → no generan ningún ingreso (TotalPrice = 0)
 - CustomerID = NaN   → no tienen cliente asociado
No son recuperables y solo añadirían ruido al modelo.

In [ ]:
print("\n--- 3.1 Eliminar filas con Description nula ---")

antes = len(df_clean)
df_clean = df_clean.dropna(subset=['Description']).reset_index(drop=True)
eliminadas = antes - len(df_clean)

print(f"  Filas antes:     {antes:,}")
print(f"  Filas eliminadas: {eliminadas:,}")
print(f"  Filas después:   {len(df_clean):,}")
print(f"  Verificación — Description nulos restantes: {df_clean['Description'].isnull().sum()}")

### 3.2 ELIMINAR DUPLICADOS EXACTOS

 Una fila duplicada exacta tiene TODOS sus campos idénticos: mismo InvoiceNo,
 StockCode, Description, Quantity, UnitPrice, InvoiceDate, CustomerID y Country.
 Esto es físicamente imposible en un sistema transaccional real: si el mismo cliente
 compra el mismo producto en el mismo instante, el sistema generaría un InvoiceNo
 distinto o un timestamp diferente. Su presencia indica errores de doble inserción
 en la BBDD, exports corruptos o fallos de ETL.

In [ ]:
print("\n--- 3.2 Eliminar duplicados exactos ---")

antes = len(df_clean)
df_clean = df_clean.drop_duplicates(keep='first', ignore_index=True)
eliminadas = antes - len(df_clean)

print(f"  Filas antes:      {antes:,}")
print(f"  Filas eliminadas: {eliminadas:,}")
print(f"  Filas después:    {len(df_clean):,}")
print(f"  Verificación — duplicados restantes: {df_clean.duplicated().sum()}")


### 3.3 ELIMINAR NEGATIVOS HUÉRFANOS

 Son filas con Quantity < 0 pero SIN prefijo "C" en InvoiceNo.
 El análisis directo del CSV confirma que el 100% cumple simultáneamente:
   - UnitPrice = 0.0  → TotalPrice = 0 siempre, sin impacto en ingresos
   - CustomerID = NaN → ninguna tiene cliente asociado
   - InvoiceNo sin "C" → el sistema nunca las registró como cancelación 

In [ ]:
print("\n--- 3.3 Eliminar negativos huérfanos (ajustes de almacén) ---")

antes = len(df_clean)
mask_huerfanos = (
    ~df_clean['InvoiceNo'].str.startswith('C', na=False) &
    (df_clean['Quantity'] < 0) &
    (df_clean['UnitPrice'] == 0.0)
)
df_clean = df_clean[~mask_huerfanos].reset_index(drop=True)
eliminadas = antes - len(df_clean)

print(f"  Filas antes:      {antes:,}")
print(f"  Filas eliminadas: {eliminadas:,}")
print(f"  Filas después:    {len(df_clean):,}")
print(f"  Verificación — negativos huérfanos restantes: {(~df_clean['InvoiceNo'].str.startswith('C', na=False) & (df_clean['Quantity'] < 0) & (df_clean['UnitPrice'] == 0.0)).sum()}")

### 3.4 ELIMINAR STOCKCODES NO ESTÁNDAR

In [ ]:
print("\n--- 3.4 Eliminar StockCodes no estándar ---")

antes = len(df_clean)
mask_std = df_clean['StockCode'].str.match(r'^[0-9]{5}[A-Za-z]?$', na=False)
df_clean = df_clean[mask_std].reset_index(drop=True)
eliminadas = antes - len(df_clean)

print(f"  Filas antes:      {antes:,}")
print(f"  Filas eliminadas: {eliminadas:,}")
print(f"  Filas después:    {len(df_clean):,}")
n_no_std = (~df_clean['StockCode'].str.match(r'^[0-9]{5}[A-Za-z]?$', na=False)).sum()
print(f"  Verificación — StockCodes no estándar restantes: {n_no_std}")

### 3.5 CAPPING (WINSORIZACIÓN) DE OUTLIERS EN Quantity Y UnitPrice


In [ ]:
print("\n--- 3.5 Capping de outliers (winsorización al percentil 99) ---")

cap_qty   = df_clean.loc[df_clean['Quantity']  > 0, 'Quantity'].quantile(0.99)
cap_price = df_clean.loc[df_clean['UnitPrice'] > 0, 'UnitPrice'].quantile(0.99)

print(f"  Umbral Quantity   (p99): {cap_qty:.1f} uds  →  clip [{-cap_qty:.1f}, {cap_qty:.1f}]")
print(f"  Umbral UnitPrice  (p99): £{cap_price:.2f}  →  clip [-, {cap_price:.2f}]")

n_qty_sup  = (df_clean['Quantity']  >  cap_qty).sum()
n_qty_inf  = (df_clean['Quantity']  < -cap_qty).sum()
n_price_sup = (df_clean['UnitPrice'] >  cap_price).sum()
print(f"  Filas Quantity  > +umbral (recortadas arriba): {n_qty_sup:,}")
print(f"  Filas Quantity  < -umbral (recortadas abajo):  {n_qty_inf:,}")
print(f"  Filas UnitPrice > umbral  (recortadas arriba): {n_price_sup:,}")

df_clean['Quantity']  = df_clean['Quantity'].clip(lower=-cap_qty, upper=cap_qty)
df_clean['UnitPrice'] = df_clean['UnitPrice'].clip(upper=cap_price)

# Recalcular TotalPrice con los valores ya capeados
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

print(f"  Quantity  rango tras capping:  [{df_clean['Quantity'].min():.1f}, {df_clean['Quantity'].max():.1f}]")
print(f"  UnitPrice máxima tras capping: £{df_clean['UnitPrice'].max():.2f}")
print(f"  Filas totales (sin cambio):    {len(df_clean):,}")

print("\n  Estadísticas post-capping (validación de rangos):")
print(df_clean[['Quantity', 'UnitPrice', 'TotalPrice']].describe().round(2))

### 3.5b ELIMINAR FILAS CON UnitPrice = 0

In [ ]:
print("\n--- 3.5b Eliminar filas con UnitPrice = 0 ---")

antes = len(df_clean)
df_clean = df_clean[df_clean['UnitPrice'] > 0].reset_index(drop=True)
eliminadas = antes - len(df_clean)

print(f"  Filas antes:      {antes:,}")
print(f"  Filas eliminadas: {eliminadas:,}")
print(f"  Filas después:    {len(df_clean):,}")
print(f"  Verificación — filas con UnitPrice = 0 restantes: {(df_clean['UnitPrice'] == 0).sum()}")

filas_post_35b = len(df_clean)

### 3.5c ELIMINAR FILAS CON Quantity = 0

In [ ]:

print("\n--- 3.5c Eliminar filas con Quantity = 0 ---")
antes = len(df_clean)
df_clean = df_clean[df_clean['Quantity'] != 0].reset_index(drop=True)
eliminadas = antes - len(df_clean)
print(f"  Filas antes:      {antes:,}")
print(f"  Filas eliminadas: {eliminadas:,}")
print(f"  Filas después:    {len(df_clean):,}")
print(f"  Verificación — filas con Quantity = 0 restantes: {(df_clean['Quantity'] == 0).sum()}")

filas_post_35c = len(df_clean)

### 3.6 MANTENER CustomerID NULOS

Los ~135.080 registros sin `CustomerID` son ventas anónimas reales. Para predecir ventas diarias
(variable objetivo = `sum(TotalPrice)` por día), el `CustomerID` es irrelevante: la transacción
genera ingresos con independencia de que el cliente esté identificado o no.
Eliminarlos supondría perder ~25 % del dataset sin ningún beneficio para el modelo.

In [ ]:
print("\n--- 3.6 CustomerID nulo — decisión: conservar ---")

n_sin_cliente = df_clean['CustomerID'].isnull().sum()
pct = n_sin_cliente / len(df_clean) * 100
print(f"  Filas con CustomerID nulo: {n_sin_cliente:,} ({pct:.2f}%)")
print(f"  Decisión: se conservan — son ventas anónimas válidas para la variable objetivo")
print(f"  Filas totales sin cambio:  {len(df_clean):,}")

### 3.7 TRATAR CANCELACIONES (prefijo "C" en InvoiceNo)

Las cancelaciones tienen `Quantity < 0` → `TotalPrice < 0`.  

**No se eliminan**: al agregar por día con `groupby('Fecha')['TotalPrice'].sum()` se restan automáticamente de las ventas brutas, dando la **venta neta real** del negocio.

Eliminarlas sobreestimaría las ventas diarias y el modelo aprendería una señal irreal.

In [ ]:
print("\n--- 3.7 Cancelaciones (prefijo C) — decisión: conservar con TotalPrice negativo ---")

mask_cancel = df_clean['InvoiceNo'].str.startswith('C', na=False)
n_cancel    = mask_cancel.sum()
tp_cancel   = df_clean.loc[mask_cancel, 'TotalPrice'].sum()
print(f"  Filas de cancelación en df_clean: {n_cancel:,}")
print(f"  TotalPrice acumulado cancelaciones: £{tp_cancel:,.2f}")
print(f"  Decisión: se conservan — el TotalPrice negativo reduce el agregado diario automáticamente")
print(f"  Filas totales sin cambio: {len(df_clean):,}")

# Verificación: días con venta neta negativa
ventas_netas_dia = df_clean.groupby('Fecha')['TotalPrice'].sum()
dias_negativos   = (ventas_netas_dia < 0).sum()
print(f"  Días con venta neta negativa (devoluciones > ventas brutas): {dias_negativos}")

### 3.8 VERIFICAR INTEGRIDAD TEMPORAL TRAS LA LIMPIEZA

Confirmamos que los pasos 3.1–3.7 no han eliminado accidentalmente días completos del rango histórico ni del test set.

- **Rango esperado**: 01/12/2010 → 09/12/2011
- **Test set**: 09/11/2011 → 09/12/2011 — todos sus días deben tener datos
- **Días sin datos**: solo festivos y fines de semana (igual que en el EDA)

In [ ]:
print("\n--- 3.8 Integridad temporal tras la limpieza ---")

fechas_clean   = df_clean['Fecha'].drop_duplicates().sort_values()
fecha_min      = fechas_clean.min()
fecha_max      = fechas_clean.max()
rango_completo = pd.date_range(start=fecha_min, end=fecha_max, freq='D')
dias_sin_datos = rango_completo.difference(fechas_clean)

print(f"  Fecha mínima en df_clean: {fecha_min.date()}")
print(f"  Fecha máxima en df_clean: {fecha_max.date()}")
print(f"  Días totales en el rango: {len(rango_completo)}")
print(f"  Días con datos:           {len(fechas_clean)}")
print(f"  Días sin datos:           {len(dias_sin_datos)}")

# Verificar que el test set completo tiene datos
TEST_INICIO = pd.Timestamp('2011-11-09')
TEST_FIN    = pd.Timestamp('2011-12-09')
dias_test   = pd.date_range(start=TEST_INICIO, end=TEST_FIN, freq='D')
dias_test_sin_datos = dias_test.difference(fechas_clean)
print(f"\n  Test set ({TEST_INICIO.date()} → {TEST_FIN.date()}):")
print(f"    Días en el rango del test:      {len(dias_test)}")
print(f"    Días con datos en el test set:  {len(dias_test) - len(dias_test_sin_datos)}")
print(f"    Días SIN datos en el test set:  {len(dias_test_sin_datos)}")
if len(dias_test_sin_datos) > 0:
    print(f"    Días sin datos: {dias_test_sin_datos.strftime('%Y-%m-%d').tolist()}")
else:
    print(f"    ✓ Todos los días del test set tienen datos")

# Días sin datos en todo el rango
print(f"\n  Días sin datos en todo el rango (esperados: festivos/fines de semana):")
print(f"    {dias_sin_datos.strftime('%Y-%m-%d').tolist()}")

### 3.9 DATASET LIMPIO

Comparativa de filas eliminadas en cada paso y guardado del CSV limpio en `contenidoCSV/data_clean.csv`.

In [ ]:
print(f"\n\n{'='*60}")
print(f"  RESUMEN LIMPIEZA — COMPARATIVA POR PASO")
print(f"{'='*60}")

pasos = [
    ("Filas originales",                     filas_iniciales,                  0),
    ("3.1 Eliminar Description nula",        540_455,   filas_iniciales - 540_455),
    ("3.2 Eliminar duplicados exactos",      535_187,   540_455 - 535_187),
    ("3.3 Eliminar negativos huérfanos",     533_851,   535_187 - 533_851),
    ("3.4 Eliminar StockCodes no estándar",  531_356,   533_851 - 531_356),
    ("3.5 Capping outliers (sin eliminar)",  531_356,   0),
    ("3.5b Eliminar UnitPrice = 0",          filas_post_35b,  531_356 - filas_post_35b),
    ("3.5c Eliminar Quantity = 0",           filas_post_35c,  filas_post_35b - filas_post_35c),
    ("3.6 CustomerID nulo (conservar)",      filas_post_35c,  0),
    ("3.7 Cancelaciones (conservar)",        filas_post_35c,  0),
]

print(f"\n  {'Paso':<42} {'Filas':>8}  {'Eliminadas':>10}")
print(f"  {'-'*62}")
for nombre, filas, eliminadas in pasos:
    marca = "  " if eliminadas == 0 else "->"
    print(f"  {marca} {nombre:<40} {filas:>8,}  {eliminadas:>10,}")

filas_finales   = len(df_clean)
total_eliminado = filas_iniciales - filas_finales
pct_eliminado   = total_eliminado / filas_iniciales * 100
print(f"\n  {'TOTAL ELIMINADAS':<42} {total_eliminado:>8,}  ({pct_eliminado:.2f}%)")
print(f"  {'FILAS FINALES EN df_clean':<42} {filas_finales:>8,}")

# Guardar CSV limpio
RUTA_CLEAN = 'contenidoCSV/data_clean.csv'
df_clean.to_csv(RUTA_CLEAN, index=False, encoding='utf-8')

print(f"\n--- 3.9 Dataset limpio guardado ---")
print(f"  Ruta:         {RUTA_CLEAN}")
print(f"  Filas:        {filas_finales:,}")
print(f"  Columnas:     {df_clean.shape[1]}")
print(f"  Columnas:     {list(df_clean.columns)}")
print(f"  Memoria (MB): {df_clean.memory_usage(deep=True).sum() / 1024**2:.1f}")

In [ ]:
# Validar variable objetivo final
print(f"\n--- 3.9 Validación variable objetivo — ventas netas diarias ---")
ventas_diarias_clean = df_clean.groupby('Fecha')['TotalPrice'].sum()
print(f"  Días con datos en df_clean:  {len(ventas_diarias_clean)}")
print("")
print("  Estadísticas:")
print(ventas_diarias_clean.describe().apply(lambda x: f'    £{x:>12,.2f}'))
dias_neg  = (ventas_diarias_clean < 0).sum()
dias_cero = (ventas_diarias_clean == 0).sum()
print(f"\n  Días con ventas negativas (devoluciones > ventas): {dias_neg}  {'<-- revisar' if dias_neg > 0 else '✓ ninguno'}")
print(f"  Días con ventas = 0:                               {dias_cero}  {'<-- revisar' if dias_cero > 0 else '✓ ninguno'}")
print(f"\n  ✓ Variable objetivo lista para la sección 4 (transformación)")

# 4. TRANSFORMACIÓN DE DATOS

### 4.1 AGREGACIÓN DIARIA Y CREACIÓN DE LA VARIABLE OBJETIVO "Ventas"

In [ ]:
print("\n\n=== 4. TRANSFORMACIÓN DE DATOS ===")
print("\n--- 4.1 Agregación diaria y creación de la variable objetivo 'Ventas' ---")

df_clean['Fecha'] = pd.to_datetime(df_clean['Fecha'])

n_ventas    = (df_clean['EsCancelacion'] == False).sum()
n_cancelac  = (df_clean['EsCancelacion'] == True).sum()
print(f"\n  Filas en df_clean:       {len(df_clean):,}")
print(f"    → Ventas reales:       {n_ventas:,}")
print(f"    → Cancelaciones:       {n_cancelac:,}")
print(f"  Decisión: Ventas = venta NETA (incluye cancelaciones como TotalPrice negativo)")

ventas_netas = (
    df_clean
    .groupby('Fecha', sort=True)
    .agg(Ventas = ('TotalPrice', 'sum'))
    .reset_index()
)

df_solo_ventas = df_clean[df_clean['EsCancelacion'] == False]

features_volumen = (
    df_solo_ventas
    .groupby('Fecha', sort=True)
    .agg(
        NumTransacc      = ('TotalPrice',  'count'),
        NumPedidos       = ('InvoiceNo',   'nunique'),
        NumClientes      = ('CustomerID',  'nunique'),
        UnidadesVendidas = ('Quantity',    'sum'),
    )
    .reset_index()
)

df_daily = ventas_netas.merge(features_volumen, on='Fecha', how='left')

rango_completo = pd.date_range(
    start=df_daily['Fecha'].min(),
    end=df_daily['Fecha'].max(),
    freq='D'
)

df_daily = (
    df_daily
    .set_index('Fecha')
    .reindex(rango_completo)
    .rename_axis('Fecha')
    .fillna(0)
    .reset_index()
)

In [ ]:
print(f"\n  Rango temporal:              {df_daily['Fecha'].min().date()} → {df_daily['Fecha'].max().date()}")
print(f"  Días en el rango completo:   {len(rango_completo)}")
print(f"  Días con Ventas > 0:         {(df_daily['Ventas'] > 0).sum()}")
print(f"  Días con Ventas = 0 (hueco): {(df_daily['Ventas'] == 0).sum()}")
print(f"  Días con Ventas < 0:         {(df_daily['Ventas'] < 0).sum()}  (devoluciones > ventas ese día)")
print(f"\n  Columnas del dataframe diario: {list(df_daily.columns)}")
print(f"\n  Primeras filas:")
print(df_daily.head(10).to_string(index=False))
print(f"\n  Estadísticas de la variable objetivo 'Ventas' (£) — venta neta:")
print(df_daily['Ventas'].describe().apply(lambda x: f"    {x:>12,.2f}").to_string())
print(f"\n  ✓ df_daily creado con {len(df_daily)} filas. Listo para las transformaciones siguientes.")

### 4.2 Creación de variables categóricas de resumen diario — Producto más vendido por día, país más frecuente por día, número de clientes únicos por día.

In [ ]:
print("\n--- 4.2 Creación de variables categóricas de resumen diario ---")

producto_top = (
    df_solo_ventas
    .groupby(['Fecha', 'StockCode'], sort=False)['Quantity']
    .sum()
    .reset_index()
    .sort_values(['Fecha', 'Quantity'], ascending=[True, False])
    .groupby('Fecha', sort=True)
    .first()                          # primera fila por fecha = mayor Quantity
    .rename(columns={'StockCode': 'ProductoTopDia'})
    [['ProductoTopDia']]
    .reset_index()
)

pais_top = (
    df_solo_ventas
    .groupby(['Fecha', 'Country'], sort=False)['InvoiceNo']
    .count()
    .reset_index()
    .sort_values(['Fecha', 'InvoiceNo'], ascending=[True, False])
    .groupby('Fecha', sort=True)
    .first()
    .rename(columns={'Country': 'PaisTopDia'})
    [['PaisTopDia']]
    .reset_index()
)

df_daily = (
    df_daily
    .merge(producto_top, on='Fecha', how='left')
    .merge(pais_top,     on='Fecha', how='left')
)

df_daily['ProductoTopDia'] = df_daily['ProductoTopDia'].fillna('Sin_Actividad')
df_daily['PaisTopDia']     = df_daily['PaisTopDia'].fillna('Sin_Actividad')

In [ ]:
print(f"\n  Columnas tras 4.2: {list(df_daily.columns)}")
print(f"\n  Días sin actividad (categóricas = 'Sin_Actividad'): "
      f"{(df_daily['ProductoTopDia'] == 'Sin_Actividad').sum()}")

print(f"\n  Top 10 productos más frecuentes como 'ProductoTopDia':")
print(df_daily['ProductoTopDia'].value_counts().head(10).to_string())

print(f"\n  Top 10 países más frecuentes como 'PaisTopDia':")
print(df_daily['PaisTopDia'].value_counts().head(10).to_string())

print(f"\n  Primeras filas con nuevas columnas:")
print(df_daily[['Fecha', 'Ventas', 'ProductoTopDia', 'PaisTopDia']].head(10).to_string(index=False))
print(f"\n  ✓ Variables categóricas añadidas a df_daily.")

### 4.3 EXTRACCIÓN DE VARIABLES TEMPORALES DESDE LA FECHA

In [ ]:
print("\n--- 4.3 Extracción de variables temporales desde la fecha ---")

df_daily['DiaSemana']    = df_daily['Fecha'].dt.dayofweek          # 0=Lun, 6=Dom
df_daily['EsFinDeSemana']= df_daily['Fecha'].dt.dayofweek.isin([5, 6]).astype(int)
df_daily['Mes']          = df_daily['Fecha'].dt.month              # 1–12
df_daily['Trimestre']    = df_daily['Fecha'].dt.quarter            # 1–4
df_daily['SemanaMes']    = df_daily['Fecha'].dt.day.apply(
                               lambda d: min((d - 1) // 7 + 1, 5) # 1–5
                           )
df_daily['DiaAnio']      = df_daily['Fecha'].dt.dayofyear          # 1–365
df_daily['SemanaAnio']   = df_daily['Fecha'].dt.isocalendar().week.astype(int)  # 1–53

In [ ]:
# ── 4.3.2  Resultado ─────────────────────────────────────────────────────────

nuevas_cols = ['DiaSemana', 'EsFinDeSemana', 'Mes', 'Trimestre',
               'SemanaMes', 'DiaAnio', 'SemanaAnio']

print(f"\n  Columnas temporales añadidas: {nuevas_cols}")
print(f"\n  Columnas totales en df_daily: {list(df_daily.columns)}")

print(f"\n  Distribución de días por DiaSemana (0=Lun … 6=Dom):")
print(df_daily['DiaSemana'].value_counts().sort_index().to_string())

print(f"\n  Días marcados como fin de semana (EsFinDeSemana=1): "
      f"{df_daily['EsFinDeSemana'].sum()}")

print(f"\n  Distribución de días por Mes:")
print(df_daily['Mes'].value_counts().sort_index().to_string())

print(f"\n  Distribución de días por Trimestre:")
print(df_daily['Trimestre'].value_counts().sort_index().to_string())

print(f"\n  Primeras filas con variables temporales:")
cols_muestra = ['Fecha', 'Ventas', 'DiaSemana', 'EsFinDeSemana',
                'Mes', 'Trimestre', 'SemanaMes', 'SemanaAnio']
print(df_daily[cols_muestra].head(10).to_string(index=False))
print(f"\n  ✓ Variables temporales extraídas y añadidas a df_daily.")

---
## Resumen del EDA

| Hallazgo | Valor |
|---------|-------|
| Filas totales | ~541.909 |
| Columnas | 8 |
| Filas con `CustomerID` nulo | ~135.080 (24,9 %) |
| Filas con `Description` nula | ~1.454 (0,27 %) |
| Filas duplicadas exactas | ~5.268 (0,97 %) |
| Transacciones con `Quantity <= 0` | ~10.624 (1,96 %) |
| Transacciones con `UnitPrice <= 0` | ~2.515 (0,46 %) |
| StockCodes no estándar | ~33.000 aprox. |
| Días sin datos en el rango | fines de semana y festivos |

**Próximo paso → Sección 3: Limpieza de datos**